In [2]:
import src.model as model
import config.config as config


In [1]:
import pandas as pd

data = pd.read_csv('full_data.txt', delimiter = ',')

In [5]:
data.head(10)

,ret,prc,shrout,vol,altprc,mktcap,year,turnover,bm,evm,...,invtq,revtq,cogsq,icaptq,piq,pstkq,rectq,req,date,permno
0,0.333333,17.000,27292.0,3645.0,17.000,463.964000,1975,0.133556,1.571662,8.232906,...,0.000,129.839,0.000,1459.189,0.000,192.976,0.000,0.000,1975-01-31,10137
1,0.136564,32.250,27556.0,5523.0,32.250,888.681000,1975,0.200428,1.330664,4.210632,...,287.919,574.808,468.938,1449.329,30.171,0.000,269.838,656.000,1975-01-31,10145
2,0.215447,37.375,23853.0,6229.0,37.375,891.505875,1975,0.261141,1.094498,9.700099,...,0.000,290.650,0.000,1330.549,49.870,1.770,0.000,633.770,1975-01-31,10161
3,0.210744,36.625,26304.0,2403.0,36.625,963.384000,1975,0.091355,1.321524,4.416657,...,0.000,578.729,0.000,1522.293,74.259,74.900,0.000,808.758,1975-01-31,10225
4,0.424528,18.875,17137.0,5248.0,18.875,323.460875,1975,0.306238,1.191198,4.263680,...,0.000,225.796,0.000,427.629,23.619,0.000,0.000,200.875,1975-01-31,10233
5,0.103448,31.250,17749.0,2714.0,31.250,554.656250,1975,0.152910,1.747075,4.005132,...,522.937,726.911,2309.187,1139.322,28.685,41.538,377.299,472.704,1975-01-31,10241
6,-0.063361,85.000,45729.0,11544.0,85.000,3886.965000,1975,0.252444,0.963768,5.032263,...,0.000,1810.396,1484.593,4407.629,261.763,48.556,0.000,2379.804,1975-01-31,10604
7,0.247706,17.000,26666.0,4152.0,17.000,453.322000,1975,0.155704,1.904987,6.614935,...,276.414,322.110,281.452,978.967,11.614,0.000,109.382,535.198,1975-01-31,10364
8,0.122430,29.500,17366.0,6391.0,29.500,512.297000,1975,0.368018,2.908244,8.074533,...,48.834,336.184,1078.478,1765.924,26.064,0.000,176.318,351.551,1975-01-31,11156
9,0.148148,15.500,22091.0,2870.0,15.500,342.410500,1975,0.129917,4.086596,3.956765,...,0.000,435.493,349.800,1673.967,49.688,0.000,0.000,706.230,1975-01-31,10495


In [8]:
df = data.select_dtypes(include='string')

In [9]:
df

,rdq,date
0,1974-10-24,1975-01-31
1,1975-01-16,1975-01-31
2,1974-10-15,1975-01-31
3,1974-10-23,1975-01-31
4,1974-10-21,1975-01-31
...,...,...
165864,2024-10-30,2024-12-31
165865,2024-12-05,2024-12-31
165866,2024-10-30,2024-12-31
165867,2024-11-07,2024-12-31


In [1]:
import src.preprocess.datapull as dp
import src.config.config as cf
import tensorflow as tf
import src.env.env as en
import src.env.agent as ag
import src.train.trainingfuncs as tfu
datahandler = dp.DataHandler(cf.load_config())
obs_windows, returns = datahandler.load_data()

[DataHandler] obs_windows: (580, 50, 20, 56)
[DataHandler] returns:     (580, 50)


In [2]:
env = en.PortfolioEnv(
    obs_windows=obs_windows,
    returns=returns,
    num_envs=8
)

# ----- Agent -----
agent = ag.PortfolioAgentCritic(cf.load_config())

# ----- Optimizer -----
optimizer = tf.keras.optimizers.Adam(
    learning_rate=0.001
)

In [3]:
tfu.train(
        agent=agent,
        env=env,
        optimizer=optimizer,
        num_epochs=50, 
        rollout_len=20,
        gamma=0.999,
        sharpe_lambda=0.1,
    )

InvalidArgumentError: {{function_node __wrapped__Mul_device_/job:localhost/replica:0/task:0/device:CPU:0}} Incompatible shapes: [20,8] vs. [20,8,50] [Op:Mul] name: 

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def plot_allocations(buffer, permnos, top_n=10):
    """
    buffer.actions: list of (B, N) arrays, one per timestep
    permnos: array of asset identifiers
    top_n: only plot the top N assets by mean absolute weight
    """
    actions = np.stack(buffer.actions)  # (T, B, N)
    
    # average across environments
    weights = actions.mean(axis=1)  # (T, N)

    # pick top N assets by mean absolute allocation
    mean_abs = np.abs(weights).mean(axis=0)  # (N,)
    top_idx = np.argsort(mean_abs)[-top_n:]
    
    weights_top = weights[:, top_idx]
    labels = permnos[top_idx]

    # --- Plot 1: Stacked area chart of allocations over time ---
    fig, axes = plt.subplots(2, 1, figsize=(14, 8))

    axes[0].stackplot(range(len(weights_top)), weights_top.T, labels=labels)
    axes[0].set_title('Portfolio Allocations Over Time')
    axes[0].set_ylabel('Weight')
    axes[0].legend(loc='upper left', fontsize=7, ncol=2)
    axes[0].axhline(0, color='black', linewidth=0.5)

    # --- Plot 2: Heatmap of weights ---
    im = axes[1].imshow(weights_top.T, aspect='auto', cmap='RdYlGn', 
                         vmin=-1, vmax=1)
    axes[1].set_title('Weight Heatmap (assets x time)')
    axes[1].set_yticks(range(top_n))
    axes[1].set_yticklabels(labels, fontsize=7)
    axes[1].set_xlabel('Timestep')
    plt.colorbar(im, ax=axes[1])

    plt.tight_layout()
    plt.show()

plot_allocations(buffer, permnos=permnos, top_n=10)

In [ ]:
import configparser

# 1. Create parser and read file
config = configparser.ConfigParser()
config.read("./config/config.ini")

# 2. Access values
api_key = config["API"]["api_key"]

lookback = int(config["SREM"]["lookback"])
num_features = int(config["SREM"]["num_features"])
embed_dim = int(config["SREM"]["embed_dim"])
num_heads = int(config["SREM"]["num_heads"])
ff_dim = int(config["SREM"]["ff_dim"])
num_layers = int(config["SREM"]["num_layers"])
dropout_rate = float(config["SREM"]["dropout_rate"])

batch_size = int(config["TRAINING"]["batch_size"])
epochs = int(config["TRAINING"]["epochs"])
learning_rate = float(config["TRAINING"]["learning_rate"])

symbols = config["DATA"]["symbols"].split(",")  # convert comma-separated string to list



20
